In [1]:
from datetime import date
import hisepy
import pandas as pd
import polars as pl
import re
import os

In [2]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

### Sample metadata

In [4]:
meta_uuid = 'd82c5c42-ae5f-4e67-956e-cd3b7bf88105'

In [5]:
meta_file = hisepy.cache_files([meta_uuid])[0]

2026-04-24 13:11:30,641 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling cache_files
2026-04-24 13:11:36,901 INFO [hisepy.logging:208] logging 2541 139373092534080 Finished cache_files, success=True, time_elapsed=3.168s


In [6]:
sample_meta = pd.read_csv(meta_file)

Convert drawDate to drawYear

In [7]:
sample_meta['sample.drawYear'] = [re.sub('-.+', '', d) for d in sample_meta['sample.drawDate']]
sample_meta = sample_meta.drop(['sample.drawDate'], axis = 1)

In [8]:
sample_meta['sample.drawYear'] = sample_meta['sample.drawYear'].astype(int)

Add Age Group, Age at First Draw, and Age at Draw

In [9]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}
sample_meta['subject.ageGroup'] = [age_groups[cohort] for cohort in sample_meta['cohort.cohortGuid']]

In [10]:
sample_meta.columns

Index(['lastUpdated', 'sample.id', 'sample.bridgingControl',
       'sample.sampleKitGuid', 'sample.visitName', 'sample.visitDetails',
       'sample.daysSinceFirstVisit', 'file.id', 'file.name', 'file.batchID',
       'file.panel', 'file.pool', 'file.fileType', 'file.majorVersion',
       'subject.id', 'subject.biologicalSex', 'subject.birthYear',
       'subject.ethnicity', 'subject.partnerCode', 'subject.race',
       'subject.subjectGuid', 'cohort.cohortGuid',
       'sample.diseaseStatesRecordedAtVisit', 'file.userTags.details',
       'file.userTags.group', 'file.userTags.name', 'file.userTags.origin',
       'file.userTags.other', 'file.userTags.version', 'pbmc_sample_id',
       'sample.reference', 'sample.drawYear', 'subject.ageGroup'],
      dtype='object')

In [11]:
sample_meta['sample.subjectAgeAtDraw'] = sample_meta['sample.drawYear'] - sample_meta['subject.birthYear']

In [12]:
first_draws = (
    pl.DataFrame(sample_meta)
    .sort(pl.col('sample.daysSinceFirstVisit'))
    .group_by('subject.subjectGuid')
    .first()
    .select(['subject.subjectGuid', 'sample.subjectAgeAtDraw'])
    .rename({'sample.subjectAgeAtDraw': 'subject.ageAtFirstDraw'})
)

In [13]:
first_draws.head()

subject.subjectGuid,subject.ageAtFirstDraw
str,i64
"""BR2033""",55
"""BR1011""",31
"""BR1014""",26
"""BR2019""",57
"""BR1035""",30


In [14]:
sample_meta = pl.DataFrame(sample_meta)

In [15]:
sample_meta = sample_meta.join(first_draws, how = 'left', on = 'subject.subjectGuid')

#### Standardize column names

In [16]:
meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'sample.visitName',
    'sample.visitDetails',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit',
    'sample.diseaseStatesRecordedAtVisit'
]

In [17]:
meta = sample_meta.select(meta_cols)

In [18]:
meta.shape

(868, 15)

## Query HISE to get surveys

First, we make a dictionary (with curly braces) that defines what we want to get. In this case, we want to get samples from our study that were previously specified in our sample metadata.

Each entry in the dictionary has to be a list (square braces), even if it has a single entry.

In [19]:
query_dict = {
    'sampleKitGuid': meta['sample.sampleKitGuid'].to_list()
}

Now, we send this dictionary to HISE via hisepy

In [20]:
sample_data = hisepy.reader.read_samples(
    query_dict = query_dict,
    to_df = False
)

2026-04-24 13:12:00,844 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling read_samples
2026-04-24 13:12:00,845 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling lookup_queryable_fields
2026-04-24 13:12:04,814 INFO [hisepy.logging:208] logging 2541 139373092534080 Finished lookup_queryable_fields, success=True, time_elapsed=1.150s
2026-04-24 13:12:04,816 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling lookup_queryable_fields
2026-04-24 13:12:08,416 INFO [hisepy.logging:208] logging 2541 139373092534080 Finished lookup_queryable_fields, success=True, time_elapsed=1.298s
2026-04-24 13:12:15,208 INFO [hisepy.logging:208] logging 2541 139373092534080 Finished read_samples, success=True, time_elapsed=12.537s


### Assemble surveys from HISE data
This make a dictionary for each sampleKitGuid, and looks for non-null values in all of the results from HISE for that kit.

This is because there are multiple entries for each kit spread across HISE projects, and we're not certain that all labs are stored in all projects. This provides a way to assemble results that are stored in any project.

In [54]:
survey_list = []
i = 0
for data in sample_data:
    survey_data = {}
    kit = data['sample']['sampleKitGuid']

    if 'survey' in data.keys():
        surveys = data['survey']
        for survey in surveys:
            if survey['surveyDesignId'] == 'f66321d6-04db-47d3-9a45-a403f698860c':
                answers = survey['answers']
                if 'visit_patient_bloodpressure' in answers.keys():
                    bp = answers['visit_patient_bloodpressure']
                else:
                    bp = 'None'

                if 'visit_patient_heartrate' in answers.keys():
                    hr = answers['visit_patient_heartrate']
                else:
                    hr = 'None'
                
                survey_df = pl.DataFrame({
                    'sample.sampleKitGuid': kit,
                    'visit_patient_bloodpressure': bp,
                    'visit_patient_heartrate': hr
                })
                survey_list.append(survey_df)

In [84]:
survey_df = pl.concat(survey_list)

In [85]:
survey_df = survey_df.filter(pl.col('visit_patient_bloodpressure') != "None")

In [86]:
survey_df = survey_df.with_columns(
    pl.col('visit_patient_bloodpressure')
      .str.replace(r"/.+", "")
      .cast(int)
      .alias('visit_patient_systolic_bloodpressure'),
    pl.col('visit_patient_bloodpressure')
      .str.replace(r".+/", "")
      .cast(int)
      .alias('visit_patient_diastolic_bloodpressure')
)

In [87]:
survey_df = survey_df.with_columns(
    pl.when(
        pl.col('visit_patient_systolic_bloodpressure') < 120,
        pl.col('visit_patient_diastolic_bloodpressure') < 80
    ).then(pl.lit('Normal'))
    .when(
        pl.col('visit_patient_systolic_bloodpressure') < 130,
        pl.col('visit_patient_diastolic_bloodpressure') < 80
    ).then(pl.lit('Elevated'))
    .when(
        pl.col('visit_patient_systolic_bloodpressure') < 140 |
        pl.col('visit_patient_diastolic_bloodpressure').is_between(80, 89)
    ).then(pl.lit('High Blood Pressure (Stage 1)'))
    .when(
        pl.col('visit_patient_systolic_bloodpressure') < 180 |
        pl.col('visit_patient_diastolic_bloodpressure').is_between(90, 119)
    ).then(pl.lit('High Blood Pressure (Stage 2)'))
    .otherwise(pl.lit('Hypertensive Crisis'))
    .alias('visit_patient_bloodpressure_category')
)

In [88]:
survey_df.shape

(175, 6)

### Combine with sample metadata

In [91]:
combined_df = meta.join(survey_df, how = 'left', on = 'sample.sampleKitGuid')

In [92]:
combined_df.shape

(868, 20)

In [94]:
combined_df = combined_df.filter(
    ~pl.col('visit_patient_bloodpressure_category').is_null()
)

In [97]:
combined_df = combined_df.sort('subject.subjectGuid')

In [98]:
out_file = 'output/sound_life_bloodpressure_survey_{d}.csv'.format(d = date.today())
combined_df.write_csv(out_file)

## Upload data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [99]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life Blood Pressure Survey and Metadata {d}'.format(d = date.today())

In [100]:
search_id = element_id()
search_id

'radon-zinc-xenon'

In [102]:
in_files = [meta_uuid]
in_files

['d82c5c42-ae5f-4e67-956e-cd3b7bf88105']

In [103]:
out_files = [out_file]

In [104]:
out_files

['output/sound_life_bloodpressure_survey_2026-04-24.csv']

In [105]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

2026-04-24 14:48:38,059 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling upload_files


Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


2026-04-24 14:48:49,701 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling get_default_store
2026-04-24 14:48:52,027 INFO [hisepy.logging:208] logging 2541 139373092534080 Finished get_default_store, success=True, time_elapsed=0.365s
2026-04-24 14:49:07,549 INFO [hisepy.logging:175] logging 2541 139373092534080 Calling conda_env_builds
2026-04-24 14:49:07,550 INFO [hisepy.logging:53] utils 2541 139373092534080 Starting conda environment build validation...
2026-04-24 14:49:08,075 INFO [hisepy.logging:75] utils 2541 139373092534080 Exporting conda environment from /home/workspace/environment/pythonscrna12...
2026-04-24 14:49:11,781 INFO [hisepy.logging:88] utils 2541 139373092534080 Removing hisepy references from exported environment file...
2026-04-24 14:49:11,785 INFO [hisepy.logging:99] utils 2541 139373092534080 Creating temporary conda environment at /tmp/conda_env_test_hsdzku_f/env_f13e08489bf94a269c1d0a949604001b...
2026-04-24 14:51:33,199 INFO [hisepy.logging:119] u

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '0fa38f66-e08d-4e43-a862-c4d6a15a1fec',
 'ProcessId': '216b6f58-cf15-4fde-a320-2efabcbfef29',
 'WorkflowId': '01d98641-467e-4f02-8c2d-0e84b2027856',
 'FileIds': ['c8a7ff67-0fcd-49a1-90ad-306824bf5784']}

In [106]:
import session_info
session_info.show()

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)
